In [1]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import logging

In [2]:
def validate_file(file):

    if not file.is_file():
        logging.warning("Skipping: %s", file.name)
        return({"file": file.name, "reason": "Not a file"})
            
    if file.stat().st_size == 0:
        logging.warning("Skipping empty file: %s", file.name)
        return({"file": file.name, "reason": "Empty File"})

    if file.name.endswith("_FAILED.json"):
        logging.warning("Skipping failed file: %s", file.name)
        return({"file": file.name, "reason": "Failed File"})
        
        
    elif (((file.name[:10]+file.name[-5:])!="tfl-bikes-.json") or (len(file.name)!=33)):
        logging.error("Formatting error using %s", file.name)
        return({"file": file.name, "reason": "Invalid Naming Convention"})
        
    return None
    

In [3]:
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s: %(message)s",
    force=True,
)

bronze_path = Path.cwd()/"data"/"bronze"

dataframes = []
error_files = []
maxfiles = 20
for file in list(bronze_path.iterdir()):
    
    if len(dataframes)>=maxfiles:
        logging.info("Processed %d files", len(dataframes))
        break
        
    flag = validate_file(file)
    
    if flag is not None:
        error_files.append(flag)
        continue
    else:
        try:
            time = (file.name[10:-5])
            date_time = datetime.strptime(
                time,
                "%Y-%m-%d-H%H-M%M"
            )
        except ValueError as e:
            error_files.append({
            "file": file.name,
            "reason": f"Failed to convert datetime :{e}"
        })
            logging.warning(f"{file.name} failed to extract datetime")
            continue
        
    try:
        with file.open("r", encoding="utf-8") as f:
            json_input = json.load(f)

        if json_input == [] or json_input == {} or json_input == "":
            logging.warning("%s contains no data", file.name)
            error_files.append({"file":file.name, "reason":"File contains no data"})
            continue
    
    except json.JSONDecodeError as e:
        logging.warning(f"Skipping invalid JSON file: {file.name}")
    
        error_files.append({
            "file": file.name,
            "reason": f"Invalid JSON: {e}"
        })
        continue
    
    df = pd.json_normalize(json_input)

    required_columns = [
        "id",
        "commonName",
        "lat",
        "lon",
        "additionalProperties"
    ]

    df = df[required_columns]
    df["date_time"] = date_time
    
    dataframes.append(df)
    logging.info(f"Added {file.name} ({len(dataframes)}/20)") 

INFO: Added tfl-bikes-2026-07-24-H12-M45.json (1/20)
INFO: Added tfl-bikes-2026-07-24-H12-M50.json (2/20)
INFO: Added tfl-bikes-2026-07-24-H12-M55.json (3/20)
INFO: Added tfl-bikes-2026-07-24-H13-M00.json (4/20)
INFO: Added tfl-bikes-2026-07-24-H13-M05.json (5/20)
INFO: Added tfl-bikes-2026-07-24-H13-M10.json (6/20)
INFO: Added tfl-bikes-2026-07-24-H13-M15.json (7/20)
INFO: Added tfl-bikes-2026-07-24-H13-M20.json (8/20)
INFO: Added tfl-bikes-2026-07-24-H13-M25.json (9/20)
INFO: Added tfl-bikes-2026-07-24-H13-M30.json (10/20)
INFO: Added tfl-bikes-2026-07-24-H13-M35.json (11/20)
INFO: Added tfl-bikes-2026-07-24-H14-M00.json (12/20)
INFO: Added tfl-bikes-2026-07-24-H14-M05.json (13/20)
INFO: Added tfl-bikes-2026-07-24-H14-M10.json (14/20)
INFO: Added tfl-bikes-2026-07-24-H14-M20.json (15/20)
INFO: Added tfl-bikes-2026-07-24-H14-M25.json (16/20)
INFO: Added tfl-bikes-2026-07-24-H14-M30.json (17/20)
INFO: Added tfl-bikes-2026-07-24-H14-M35.json (18/20)
INFO: Added tfl-bikes-2026-07-24-H14-

In [4]:
df = pd.concat(dataframes, ignore_index=True)
df_long = df.explode("additionalProperties",True)
properties = pd.json_normalize(
    df_long["additionalProperties"]
)
dfx = pd.concat(
    [
        df_long[["id", "date_time"]],
        properties
    ],
    axis=1
)
dfx_filtered = dfx[["id", "date_time", "key", "value"]]
wanted = ["NbBikes", "NbStandardBikes", "NbEBikes", "NbEmptyDocks", "NbDocks"]

In [5]:
wanted_rows = dfx_filtered[dfx_filtered["key"].isin(wanted)]

In [6]:
df_pivot = wanted_rows.pivot(index=["id", "date_time"], columns="key", values="value")
wanted_columns = ['NbBikes', 'NbEBikes', 'NbStandardBikes', 'NbDocks', 'NbEmptyDocks']

new_df = df_pivot.loc[:, wanted_columns].copy()

new_df.dtypes

new_df = new_df.astype("Int64")

new_df.head(10)

key                               NbBikes  NbEBikes  NbStandardBikes  NbDocks  \
id           date_time                                                          
BikePoints_1 2026-07-24 12:45:00        2         1                1       19   
             2026-07-24 12:50:00        3         2                1       19   
             2026-07-24 12:55:00        3         2                1       19   
             2026-07-24 13:00:00        3         2                1       19   
             2026-07-24 13:05:00        3         2                1       19   
             2026-07-24 13:10:00        3         2                1       19   
             2026-07-24 13:15:00        3         2                1       19   
             2026-07-24 13:20:00        3         2                1       19   
             2026-07-24 13:25:00        3         2                1       19   
             2026-07-24 13:30:00        3         2                1       19   

key                               NbEmptyDocks  
id           date_time                          
BikePoints_1 2026-07-24 12:45:00            16  
             2026-07-24 12:50:00            16  
             2026-07-24 12:55:00            16  
             2026-07-24 13:00:00            16  
             2026-07-24 13:05:00            16  
             2026-07-24 13:10:00            16  
             2026-07-24 13:15:00            16  
             2026-07-24 13:20:00            16  
             2026-07-24 13:25:00            16  
             2026-07-24 13:30:00            16

In [7]:
sucessful_no = len(dataframes)
error_no = len(error_files)

print(f"Successfully configured {sucessful_no}/{sucessful_no+error_no} files")
for i in error_files:
    file = i.get("file")
    reason = i.get("reason")
    print(f"{file} -> {reason}")

Successfully configured 20/26 files
tfl-bikes-2026-07-24-H13-M40.json -> File contains no data
tfl-bikes-2026-07-24-H13-M45.json -> Empty File
tfl-bikes-2026-07-24-H13-M50.json -> Empty File
tfl-bikes-2026-07-24-H13-M55.json -> Empty File
tfl-bikes-2026-07-24-H13-M56_FAILED.json -> Failed File
tfl-bikes-2026-07-24-H14-M15.json -> Empty File
